In [ ]:
# !pip install transformers datasets scikit-learn pandas

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("./LIAR/combined liar and renew/liar and renew training.csv")

assert (
    "label" in df.columns and "statement" in df.columns
), "CSV must have 'label' and 'statement' columns"

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["statement"].tolist(), df["label"].tolist(), test_size=0.2, random_state=42
)

print(df.head())

                                           statement  label
0  Things went wrong with the Medicare prescripti...      1
1  I believe in a woman's right to choose. I alwa...      0
2  There are actually only 30 countries that prac...      1
3  Discretionary spending went up 84 percent in t...      0
4  To give the proposed economic stimulus plan so...      1


In [ ]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)

/Users/albertdelgado/miniconda3/envs/fake-news-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch


class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx])
                for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)

In [ ]:
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

training_args = TrainingArguments(
    output_dir="./distilbert_liar_renew_model",  
    evaluation_strategy="epoch",  
    save_strategy="epoch",  
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/albertdelgado/miniconda3/envs/fake-news-env/lib/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score
from transformers import Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.692300,0.622186,0.654921
2,0.593400,0.630163,0.652896
3,0.499700,0.693879,0.629000
4,0.237500,0.781290,0.640340


TrainOutput(global_step=2472, training_loss=0.4881479665395897, metrics={'train_runtime': 6426.6963, 'train_samples_per_second': 6.146, 'train_steps_per_second': 0.385, 'total_flos': 5232462246912000.0, 'train_loss': 0.4881479665395897, 'epoch': 4.0})

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

predictions_output = trainer.predict(val_dataset)
preds = np.argmax(predictions_output.predictions, axis=1)
labels = predictions_output.label_ids

accuracy = accuracy_score(labels, preds)
precision = precision_score(labels, preds)
recall = recall_score(labels, preds)
f1 = f1_score(labels, preds)

tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
specificity = tn / (tn + fp)

print(f"Accuracy:     {accuracy:.4f}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"Specificity:  {specificity:.4f}")
print(f"F1 Score:     {f1:.4f}")

Accuracy:     0.6549
Precision:    0.5647
Recall:       0.4749
Specificity:  0.7687
F1 Score:     0.5159


In [ ]:
save_directory = "./fine tuned/distilbert_liar_renew_model"

model.save_pretrained(save_directory)

tokenizer.save_pretrained(save_directory)

print(f"Model and tokenizer saved to {save_directory}")

Model and tokenizer saved to ./fine tuned/distilbert_liar_renew_model


Testing distilbert fine tuned in the LIAR+R (the new dataset that has both training files together) in LIAR test


In [ ]:
import pandas as pd

liar_df_test = pd.read_csv("./LIAR/cleaned for training/cleaned_test.csv")

assert (
    "label" in liar_df_test.columns and "statement" in liar_df_test.columns
), "Test CSV must have 'label' and 'statement' columns"

test_texts = liar_df_test["statement"].tolist()
test_labels = liar_df_test["label"].tolist()

test_encodings = tokenizer(test_texts, truncation=True, padding=True)

test_dataset = NewsDataset(test_encodings, test_labels)

In [15]:
print(liar_df_test.tail(20))

                                              statement  label
1247  Says Barack Obama promised to halve the defici...      1
1248  Says the government has gotten the TARP money ...      0
1249  Points of Light is the worlds largest voluntee...      0
1250  Says Mark Pryorcut Medicare to pay for Obamacare.      0
1251  Says Tim Kaine actually tried to raise taxes b...      1
1252  Says Marco Rubio is proposing a new $1 trillio...      1
1253  I am the only senator who turned down the stat...      1
1254                      Female buffalo lead the herd.      1
1255  There is no system to vet refugees from the Mi...      0
1256  Says Chris Christies plan to kick-start our ec...      0
1257  Obama used $20 million in federal money to emm...      0
1258                              On offshore drilling.      0
1259  We came out of the White House not only dead b...      0
1260  I think its seven or eight of the California s...      0
1261  Sen. Bob Menendez voted to enact a new tax on ...

In [ ]:
from transformers import DistilBertForSequenceClassification

model_path = "./fine tuned/distilbert_liar_renew_model"
model = DistilBertForSequenceClassification.from_pretrained(model_path)

trainer = Trainer(model=model)

test_output = trainer.predict(test_dataset)
test_preds = np.argmax(test_output.predictions, axis=1)
test_true = test_output.label_ids

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

accuracy = accuracy_score(test_true, test_preds)
precision = precision_score(test_true, test_preds)
recall = recall_score(test_true, test_preds)
f1 = f1_score(test_true, test_preds)

tn, fp, fn, tp = confusion_matrix(test_true, test_preds).ravel()
specificity = tn / (tn + fp)

print("=== Test Set Evaluation ===")
print(f"Accuracy:     {accuracy:.4f}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"Specificity:  {specificity:.4f}")
print(f"F1 Score:     {f1:.4f}")

=== Test Set Evaluation ===
Accuracy:     0.6567
Precision:    0.5165
Recall:       0.4878
Specificity:  0.7494
F1 Score:     0.5017


Testing distilbert fine tuned in the LIAR+R (the new dataset that has both training files together) in ReNew Test


In [ ]:
renew_df_test = pd.read_csv("./ReNew/combined renew test.csv")

assert (
    "label" in renew_df_test.columns and "statement" in renew_df_test.columns
), "CSV must have 'label' and 'statement' columns"

renew_texts = renew_df_test["statement"].tolist()
renew_labels = renew_df_test["label"].tolist()

renew_encodings = tokenizer(renew_texts, truncation=True, padding=True)

renew_test_dataset = NewsDataset(renew_encodings, renew_labels)

In [ ]:
renew_output = trainer.predict(renew_test_dataset)
renew_preds = np.argmax(renew_output.predictions, axis=1)
renew_true = renew_output.label_ids

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

accuracy = accuracy_score(renew_true, renew_preds)
precision = precision_score(renew_true, renew_preds)
recall = recall_score(renew_true, renew_preds)
f1 = f1_score(renew_true, renew_preds)

tn, fp, fn, tp = confusion_matrix(renew_true, renew_preds).ravel()
specificity = tn / (tn + fp)

print("=== ReNew Test Set Evaluation ===")
print(f"Accuracy:     {accuracy:.4f}")
print(f"Precision:    {precision:.4f}")
print(f"Recall:       {recall:.4f}")
print(f"Specificity:  {specificity:.4f}")
print(f"F1 Score:     {f1:.4f}")

=== ReNew Test Set Evaluation ===
Accuracy:     0.7154
Precision:    0.8750
Recall:       0.5038
Specificity:  0.9278
F1 Score:     0.6394
